# Data engine: from a search to a raw data node

The data engine finds a dataset at a partner repository, downloads it into the
**data store**, records what arrived, and writes the **raw data node**: the
starting node of any workflow that uses that data. It never reimplements a
transfer: each partner is reached through its own client.

| Source | Repository | Client | Login |
|---|---|---|---|
| `cbs_public` | RIKEN CBS (neurodata.riken.jp), published datasets | MDRS client | none |
| `cbs_project` | RIKEN CBS, everything the shared account sees (incl. BM2.0 internal data, no DOI needed) | MDRS client | `NW_MDRS_USER`, `NW_MDRS_PASSWORD` |
| `atr` | ATR Aspera share (bmb-asp.atr.jp) | aspera-client | `NW_ASPERA_USER`, `NW_ASPERA_PASSWORD` |

Steps, same shape as the optimization engine (`build_spec` → `summary()` → `optimize`):

1. **search**: text → datasets
2. **browse**: what is inside a dataset, with sizes
3. **request**: choose what to download → `summary()`; nothing is downloaded yet
4. **fetch**: download + record
5. **write the raw node**, and run it in a workflow
6. **ensure**: what the node calls: the local path, checked complete

## Install (once per machine)

The engine downloads through each partner's own client, so it needs them installed
in **this kernel's** Python (`%pip` installs there):

| What | Needed for | Install |
|---|---|---|
| [mdrs-client-python](https://git.ni.riken.jp/niu/mdrs-client-python) (RIKEN NIU) | `cbs_public`, `cbs_project`: search, browse, download | `%pip install "git+https://git.ni.riken.jp/niu/mdrs-client-python.git"` |
| [aspera-client](https://github.com/orrisroot/aspera-client) ([API reference](https://github.com/orrisroot/aspera-client/blob/main/API_REFERENCE.md)) | `atr`: search, browse, download | `%pip install "git+https://github.com/orrisroot/aspera-client"`, then `aspera setup` (installs the client's transfer program) |

Run the cell below, then **restart the kernel**.

In [ ]:
#%pip install "git+https://git.ni.riken.jp/niu/mdrs-client-python.git"
#%pip install "git+https://github.com/orrisroot/aspera-client"

In [ ]:
#!aspera setup

`aspera setup` installs `ascp`, the program that runs the ATR downloads. On Linux the cell above is all that is needed.

**Mac with Apple chip:** if `aspera setup` ends with `SDK installation failed: not a gzip file`, install `ascp` by hand with the four cells below (IBM ships the Mac version as a `.zip`, which the client cannot unpack yet). Skip them otherwise.

1. Make a folder for it:

In [ ]:
#!mkdir -p ~/aspera_sdk

2. Download IBM's transfer SDK for Mac (Apple chip), the same file `aspera setup` downloads:

In [ ]:
#!curl -L -o ~/aspera_sdk/ibm-aspera-transfer-sdk-macos-arm64-1.1.9.zip https://production-transfer-sdk.s3.us-west-2.amazonaws.com/1.1.9/ibm-aspera-transfer-sdk-macos-arm64-1.1.9.zip

3. Unzip it:

In [ ]:
#!unzip -o -q ~/aspera_sdk/ibm-aspera-transfer-sdk-macos-arm64-1.1.9.zip -d ~/aspera_sdk

4. Check that `ascp` runs (it prints its version). The ATR section below uses it from this place:

In [ ]:
#!~/aspera_sdk/ibm-aspera-transfer-sdk-macos-arm64-1.1.9/bin/ascp -A

In [ ]:
import os
import sys

# Prefer this repository's sources over any previously installed copy:
# neuroworkflow.dataengine is new and an older install would not have it.
sys.path.insert(0, os.path.abspath('../src'))

from neuroworkflow.dataengine import DataEngine, ensure, load_env, read_record

## Credentials

Kept in `gui/.env`, next to the other keys of the app, never in a notebook:

```
NW_MDRS_USER=...
NW_MDRS_PASSWORD=...
NW_ASPERA_USER=...
NW_ASPERA_PASSWORD=...
NW_ASPERA_URL=https://bmb-asp.atr.jp/
```

`load_env` sets only the `NW_*` names in this session, so the LLM keys in the same
file stay out of it. It prints the names it loaded, never the values.

In [ ]:
print('loaded:', load_env('../gui/.env'))

## The data store

Every dataset lands in `<store>/<repository>/<dataset id>/`, with a record file
`.dataset.json` beside the data. `cbs_public` and `cbs_project` share the `cbs/`
folder, so a dataset is never downloaded twice. The store is the `store`
argument, else the `NW_DATA_STORE` environment variable, else `./data`. Here it
is under `results/`, which git ignores.

`ready()` tells which sources can be used here: a source whose credentials are
missing is skipped by `search`.

In [ ]:
STORE = os.path.abspath('./results/datastore')

eng = DataEngine(store=STORE)
print('data store:', eng.store)
print('ready     :', eng.ready())

## 1. Search

Text in, datasets out, from every ready source (or `sources=[...]`). Each hit
shows its size and access level; sizes can be hundreds of GB, which is why step 2
exists.

`cbs_public` searches the published datasets (title, laboratory, creators).
`cbs_project` searches the folder names and descriptions of every laboratory's
repository, so it finds internal datasets without a DOI; it reads about 60
laboratories once per session, so its first search takes longer.

In [ ]:
# The text here is typed by hand. In the app it will come from the catalog: the
# catalog search returns matching records, and the name (or keywords) of the
# record the user picks becomes this text, to locate the dataset at its source.
hits = eng.search('marmoset', sources=['cbs_public'])
eng.show(hits)

A dataset can also be named directly, when the reference is already known:
a CBS DOI or folder UUID, or an ATR path. The folder UUID is what the engine
keeps for CBS: every CBS folder has one, with or without a DOI, and it does not
change when the folder is moved.

In [ ]:
# The reference here is typed by hand. In the app it will come from the catalog
# when its record carries one: a catalog hit's `dataset_doi` gives {'doi': ...};
# a CBS folder UUID gives {'folder_id': ...}. Without one, use search() above.
ds = eng.resolve('cbs_public', {'doi': '10.60178/cbs.20260602-001'})
print(ds.line())
print('reference kept:', ds.ref)

## 2. Browse

The sub-folders and files directly inside the dataset, with sizes and access
level. Choose from these by name.

In [ ]:
eng.show_items(eng.browse(ds))

## 3. Request: look before downloading

`request()` checks the names exist and adds up the size. Nothing is downloaded.
An empty `include` means the whole dataset.

In [ ]:
req = eng.request(ds, include=['ANTs_registration_scripts',
                               'Calcium_imaging_correlation_Analysis'])
print(req.summary())

## 4. Fetch

Downloads the request and records each item as it completes. Running it again
downloads nothing: items already recorded as complete are skipped.

In [ ]:
res = eng.fetch(req)
print(res)

for name, entry in res.record['items'].items():
    print(f"{name:<40} complete={entry['complete']}  "
          f"{entry['local_size']} of {entry['remote_size']} bytes")

In [ ]:
import time

t = time.time()
print(eng.fetch(req), f'({time.time() - t:.2f} s: nothing downloaded again)')

The record (`.dataset.json`) carries where the data came from, its access
level, the metadata given by the source, and which items finished. A download cut
halfway leaves files that look finished; the record is how a reader tells the
difference.

In [ ]:
rec = read_record(res.path)
for k in ('source', 'ref', 'title', 'doi', 'laboratory', 'access', 'size'):
    print(f'{k:<11}: {rec[k]}')

## 5. The raw data node

The engine writes the node for what was downloaded:

- **parameters**: `source` + `reference` + `include` (where the data is), `metadata`
  (as given by the source; data owners may complete it), `download` (off)
- **outputs**: `raw_data_path`, `files`, `metadata`
- **method** `load`: checks the data is complete here and passes it on

Preprocessing is not in this node: it belongs in nodes connected after it (written
by hand or by our agents), which take `raw_data_path` or `files` as input. The file
is a normal node: upload it to the app under **data**.

In [ ]:
NODES = os.path.abspath('./results/data_nodes')   # node files (.py) go here; the data stays in the store

node_file = res.write_node(NODES)
print(node_file)
print(open(node_file).read()[:1500], '...')

In [ ]:
import importlib

from neuroworkflow import WorkflowBuilder

sys.path.insert(0, NODES)
class_name = os.path.basename(node_file)[:-3]
RawNode = getattr(importlib.import_module(class_name), class_name)

raw = RawNode('RawData')
wf = WorkflowBuilder('RawDataWorkflow')
wf.add_node(raw)
wf.context['data_store'] = STORE        # where the node looks for the data
workflow = wf.build()
assert workflow.execute(), 'workflow.execute() returned False: a node failed'

for pname, port in raw._output_ports.items():
    print(pname, 'OK' if port.value is not None else '*** None ***')
print('\nraw_data_path:', raw._output_ports['raw_data_path'].value)
for f in raw._output_ports['files'].value:
    print(f"  {f['path']:<75} {f['size_bytes']:>7} B")

## 6. Ensure: what the node calls

`ensure()` returns the local folder of a dataset, checked complete against the
record **and** the disk. By default it downloads nothing: inside a workflow, a
dataset that is not here fails with a clear message instead of starting a large
transfer mid-run. `download=True` fetches the missing items first.

In [ ]:
print(ensure('cbs_public', ds.ref, include=['ANTs_registration_scripts'], store=STORE))

In [ ]:
# Not downloaded yet -> refused, nothing fetched
try:
    ensure('cbs_public', ds.ref, include=['Supplementary Figure 1'], store=STORE)
except FileNotFoundError as e:
    print('refused:', e)

## CBS project data (logged in)

With the shared account, `cbs_project` searches every laboratory's repository
folders, not only the published ones. The access level of each hit says who may
open it (`Public`, `Storage`, `CBS Open`, `PW Open`, ...). The account can list
them all, but it can open and download only some: a `PW Open` folder needs its own
password. Runs only when `NW_MDRS_USER` / `NW_MDRS_PASSWORD` are loaded.

In [ ]:
if eng.ready()['cbs_project']:
    project_hits = eng.search('marmoset', sources=['cbs_project'])
    eng.show(project_hits)
    not_public = [h for h in project_hits if h.access != 'Public']
    print(f'\n{len(not_public)} of {len(project_hits)} hits are not public')
else:
    print('cbs_project not ready: set NW_MDRS_USER / NW_MDRS_PASSWORD in gui/.env')

### Download without a DOI

A DOI is not needed: every hit carries its own reference, `ref`, which is the
folder UUID at CBS (every folder has one) and the path on the ATR share. The
engine uses that reference for everything that follows.

In [ ]:
DOWNLOAD_CBS_PROJECT = False

if eng.ready()['cbs_project'] and project_hits:
    ref = project_hits[0].ref       # {'folder_id': '...'}; later given by the catalog
    print('reference:', ref)
    try:
        p_ds = eng.resolve('cbs_project', ref)
        p_items = eng.browse(p_ds)
        eng.show_items(p_items)
        p_req = eng.request(p_ds, include=[p_items[0].name])
        print(p_req.summary())
        if DOWNLOAD_CBS_PROJECT:
            print(eng.fetch(p_req))
    except PermissionError as e:
        print('not open to this account:', e)

## ATR Aspera share

An Aspera share has paths, no ids and no metadata: a dataset is a folder, named
by its path, and search matches folder names. Downloads need IBM's `ascp`
(install cell at the top). Runs only when `NW_ASPERA_USER` / `NW_ASPERA_PASSWORD` are loaded; nothing
is downloaded unless `DOWNLOAD_ATR` is set.

In [ ]:
# ascp installed by hand (Mac cells at the top), if that was used
manual_ascp = os.path.expanduser('~/aspera_sdk/ibm-aspera-transfer-sdk-macos-arm64-1.1.9/bin/ascp')
if os.path.isfile(manual_ascp):
    os.environ['ASPERA_ASCP'] = manual_ascp

DOWNLOAD_ATR = False

if eng.ready()['atr']:
    atr_hits = eng.search('', sources=['atr'])       # every folder in the top levels
    eng.show(atr_hits)
else:
    print('atr not ready: set NW_ASPERA_USER / NW_ASPERA_PASSWORD in gui/.env')

In [ ]:
if eng.ready()['atr']:
    atr_ds = eng.resolve('atr', {'path': '/SRPBS_OPEN'})   # the path is the reference; later given by the catalog
    atr_items = eng.browse(atr_ds)
    eng.show_items(atr_items)
    smallest = min((i for i in atr_items if i.kind == 'file'), key=lambda i: i.size, default=None)
    if smallest:
        atr_req = eng.request(atr_ds, include=[smallest.name])
        print(atr_req.summary())
        if DOWNLOAD_ATR:
            print(eng.fetch(atr_req))